# Business Metrics Cheat Sheet
Reference notebook for common product analytics metrics: definitions, SQL patterns, and common mistakes to avoid.

---

## 1. User Activity

### MAU — Monthly Active Users
**Definition:** Distinct users active in a 28-day rolling window.

**Pitfalls:**
- Use a fixed 28-day window, not a calendar month — month length varies and makes comparisons uneven
- Exclude test/bot users and internal accounts
- Never count on an incomplete period (e.g. mid-month snapshot)

In [ ]:
mau_query = """
SELECT
  COUNT(DISTINCT user_guid) AS mau
FROM your_events_table
WHERE event_date >= DATE_SUB(CURRENT_DATE(), 28)
  AND event_date <  CURRENT_DATE()
"""
print(mau_query)

### WAU — Weekly Active Users
**Definition:** Distinct users active in a 7-day window, aligned to a fiscal week.

**Pitfalls:**
- Never query a partial week — filter to `fiscal_wk_ending_date <= last_friday_dt`
- Align to fiscal week boundaries for consistent comparisons
- Do not sum WAU across use cases or apps — same user appears in multiple rows, causing overcount

In [ ]:
wau_query = """
WITH cte_params AS (
  SELECT
    DATE_SUB(
      CURRENT_DATE(),
      PMOD(DAYOFWEEK(CURRENT_DATE()) - 6, 7)
    ) AS last_friday_dt
)
SELECT
  fiscal_wk_ending_date,
  COUNT(DISTINCT user_guid) AS wau
FROM your_rollup_table
CROSS JOIN cte_params
WHERE fiscal_wk_ending_date <= last_friday_dt
GROUP BY fiscal_wk_ending_date
ORDER BY fiscal_wk_ending_date DESC
"""
print(wau_query)

### DAU — Daily Active Users
**Definition:** Distinct users active on a given calendar day.

**Pitfalls:**
- Day-of-week effects are significant — compare Monday to Monday, not Monday to Sunday
- Distinguish `event_date` (when event happened) from `etl_date` (when data landed) — data may arrive late

In [ ]:
dau_query = """
SELECT
  event_date,
  COUNT(DISTINCT user_guid) AS dau
FROM your_events_table
WHERE event_date = '2026-03-03'
GROUP BY event_date
"""
print(dau_query)

---
## 2. Retention & Returning Users

### RMAU — Returning Monthly Active Users
**Definition:** Users active this 28-day window who were also active in the prior 28-day window.

**Pitfalls:**
- Lookback must be anchored to exactly 28 days, not 'last month'
- The first week in the dataset always shows 0 RMAU — exclude from trend analysis
- Summing RMAU across use cases overcounts (same user in multiple rows)

In [ ]:
rmau_query = """
SELECT
  fiscal_wk_ending_date,
  COUNT(DISTINCT user_guid)                                           AS mau,
  COUNT(DISTINCT CASE WHEN is_returning_user = 1 THEN user_guid END) AS rmau
FROM your_rollup_table
GROUP BY fiscal_wk_ending_date
ORDER BY fiscal_wk_ending_date DESC
"""
print(rmau_query)

### Retention Rate
**Definition:** Share of MAU that is returning (RMAU / MAU × 100).

**Pitfalls:**
- New product launches or onboarding spikes inflate MAU without inflating RMAU — retention rate drops artificially; annotate before drawing conclusions
- Always use NULLIF to avoid divide-by-zero

In [ ]:
retention_rate_query = """
SELECT
  fiscal_wk_ending_date,
  COUNT(DISTINCT user_guid)                                           AS mau,
  COUNT(DISTINCT CASE WHEN is_returning_user = 1 THEN user_guid END) AS rmau,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN is_returning_user = 1 THEN user_guid END)
    / NULLIF(COUNT(DISTINCT user_guid), 0), 1
  ) AS retention_rate_pct
FROM your_rollup_table
GROUP BY fiscal_wk_ending_date
ORDER BY fiscal_wk_ending_date DESC
"""
print(retention_rate_query)

### D1 / D7 / D30 Cohort Retention
**Definition:** Share of users who return exactly 1, 7, or 30 days after their first event.

**Pitfalls:**
- Incomplete cohorts: users acquired in the last 30 days cannot yet have a D30 value — exclude them from D30 calculations
- Use the user's actual first-seen date as the cohort anchor, not event_date

In [ ]:
cohort_retention_query = """
WITH first_seen AS (
  SELECT
    user_guid,
    MIN(DATE(event_date)) AS first_event_date
  FROM your_events_table
  GROUP BY user_guid
),
cohort AS (
  SELECT
    f.user_guid,
    f.first_event_date,
    MAX(CASE WHEN DATE(e.event_date) = DATE_ADD(f.first_event_date, 1)  THEN 1 ELSE 0 END) AS returned_d1,
    MAX(CASE WHEN DATE(e.event_date) = DATE_ADD(f.first_event_date, 7)  THEN 1 ELSE 0 END) AS returned_d7,
    MAX(CASE WHEN DATE(e.event_date) = DATE_ADD(f.first_event_date, 30) THEN 1 ELSE 0 END) AS returned_d30
  FROM first_seen f
  LEFT JOIN your_events_table e ON e.user_guid = f.user_guid
  WHERE f.first_event_date <= DATE_SUB(CURRENT_DATE(), 30)  -- exclude incomplete D30 cohorts
  GROUP BY f.user_guid, f.first_event_date
)
SELECT
  first_event_date,
  COUNT(DISTINCT user_guid)                        AS cohort_size,
  ROUND(100.0 * SUM(returned_d1)  / COUNT(*), 1)  AS d1_retention_pct,
  ROUND(100.0 * SUM(returned_d7)  / COUNT(*), 1)  AS d7_retention_pct,
  ROUND(100.0 * SUM(returned_d30) / COUNT(*), 1)  AS d30_retention_pct
FROM cohort
GROUP BY first_event_date
ORDER BY first_event_date DESC
"""
print(cohort_retention_query)

---
## 3. Churn & Growth

### Churn Rate
**Definition:** Share of active users from the prior period who did not return this period.

**Pitfalls:**
- Distinguish user churn from revenue churn — a churned free user is not the same as a churned paid user
- Do not measure churn on too short a window — users skipping one week are not necessarily churned
- Separate voluntary churn (cancellation) from involuntary (payment failure)

In [ ]:
churn_query = """
WITH weekly AS (
  SELECT
    fiscal_wk_ending_date,
    COUNT(DISTINCT user_guid)                                           AS mau,
    COUNT(DISTINCT CASE WHEN is_returning_user = 1 THEN user_guid END) AS rmau
  FROM your_rollup_table
  GROUP BY fiscal_wk_ending_date
)
SELECT
  fiscal_wk_ending_date,
  mau,
  rmau,
  LAG(mau) OVER (ORDER BY fiscal_wk_ending_date)  AS mau_prev,
  ROUND(
    100.0 * (LAG(mau) OVER (ORDER BY fiscal_wk_ending_date) - rmau)
    / NULLIF(LAG(mau) OVER (ORDER BY fiscal_wk_ending_date), 0), 1
  ) AS churn_rate_pct
FROM weekly
ORDER BY fiscal_wk_ending_date DESC
"""
print(churn_query)

### WoW / MoM Growth
**Definition:** Week-over-week or month-over-month % change in a metric.

**Pitfalls:**
- Holiday weeks, launches, or outages create artificial spikes — always annotate before drawing conclusions
- Never compare against an incomplete prior period
- LAG() breaks silently if there are gaps in the data — verify continuity first

In [ ]:
wow_growth_query = """
WITH weekly AS (
  SELECT
    fiscal_wk_ending_date,
    COUNT(DISTINCT user_guid) AS mau
  FROM your_rollup_table
  GROUP BY fiscal_wk_ending_date
)
SELECT
  fiscal_wk_ending_date,
  mau,
  LAG(mau) OVER (ORDER BY fiscal_wk_ending_date) AS mau_prev_week,
  ROUND(
    100.0 * (mau - LAG(mau) OVER (ORDER BY fiscal_wk_ending_date))
    / NULLIF(LAG(mau) OVER (ORDER BY fiscal_wk_ending_date), 0), 1
  ) AS wow_growth_pct
FROM weekly
ORDER BY fiscal_wk_ending_date DESC
"""
print(wow_growth_query)

---
## 4. Adoption & Engagement

### DAU / MAU Ratio (Stickiness)
**Definition:** How often monthly users return within the month. 1.0 = every day, ~0.03 = once a month.

**Pitfalls:**
- Meaningless for tools not designed for daily use (e.g. tax software, annual report builders)
- Do not mix a 7-day DAU with a 28-day MAU denominator

In [ ]:
stickiness_query = """
SELECT
  dau.event_date,
  dau.dau,
  mau.mau,
  ROUND(dau.dau / NULLIF(mau.mau, 0), 3) AS dau_mau_ratio
FROM (
  SELECT event_date, COUNT(DISTINCT user_guid) AS dau
  FROM your_events_table
  GROUP BY event_date
) dau
JOIN (
  SELECT
    event_date,
    COUNT(DISTINCT user_guid) AS mau
  FROM your_events_table
  WHERE event_date >= DATE_SUB(event_date, 28)
  GROUP BY event_date
) mau USING (event_date)
ORDER BY event_date DESC
"""
print(stickiness_query)

### Feature Adoption Rate
**Definition:** Share of active users who used a specific feature in a period.

**Pitfalls:**
- Denominator should be users who had *access* to the feature, not all MAU
- Count intentional interactions, not impressions or views

In [ ]:
feature_adoption_query = """
SELECT
  fiscal_wk_ending_date,
  COUNT(DISTINCT user_guid)                                                        AS mau,
  COUNT(DISTINCT CASE WHEN event_workflow = 'your_feature' THEN user_guid END)     AS feature_users,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN event_workflow = 'your_feature' THEN user_guid END)
    / NULLIF(COUNT(DISTINCT user_guid), 0), 1
  ) AS feature_adoption_pct
FROM your_events_table
GROUP BY fiscal_wk_ending_date
ORDER BY fiscal_wk_ending_date DESC
"""
print(feature_adoption_query)

### Events per User
**Definition:** Average engagement depth per active user in a period.

**Pitfalls:**
- Highly skewed by power users — report median or p90 alongside the mean
- Exclude duplicate or retry events to avoid inflating the count

In [ ]:
events_per_user_query = """
SELECT
  fiscal_wk_ending_date,
  COUNT(DISTINCT user_guid)                                    AS mau,
  COUNT(DISTINCT event_guid)                                   AS total_events,
  ROUND(COUNT(DISTINCT event_guid)
    / NULLIF(COUNT(DISTINCT user_guid), 0), 1)                AS avg_events_per_user
FROM your_events_table
GROUP BY fiscal_wk_ending_date
ORDER BY fiscal_wk_ending_date DESC
"""
print(events_per_user_query)

---
## 5. Cross-Cutting Pitfalls

| Area | Pitfall | Why it matters |
|---|---|---|
| Fiscal periods | Querying a period that has not yet closed | Current week numbers are always lower — filter to `fiscal_wk_ending_date <= last_friday_dt` |
| Partitioning | Filtering on non-partition columns | Always filter on the partition key (e.g. `event_date`) first to avoid full table scans |
| Grain confusion | Summing MAU across dimensions | The same user appears in multiple rows per use_case/app — always `COUNT(DISTINCT user_guid)` |
| Entitlement lag | Snapshot date mismatch | A future snapshot used to classify a past event misattributes the user's segment |
| Guest users | GUIDs with dashes are guests | Guest/anonymous users cannot be matched to entitlement data — track separately from paid MAU |